# 🔍 Multimodal Fake News Detection — Google Colab Training
**BERT + ViT + Cross-Modal Attention** framework for detecting fake news.

This notebook lets you train the full model on **Colab's free GPU**.

In [ ]:
# ============================================================
# STEP 1: Runtime Check & Setup
# ============================================================
import torch, os
USE_CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if USE_CUDA else 'cpu'

print('GPU:', torch.cuda.get_device_name(0) if USE_CUDA else 'NOT AVAILABLE (CPU runtime)')
print('CUDA:', USE_CUDA)
print('Device:', DEVICE)

PROJECT_DIR = '/content/fake_news_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
print('Project dir:', PROJECT_DIR)



In [ ]:
# ============================================================
# STEP 2: Install Dependencies
# ============================================================
!pip -q install --upgrade pip
!pip -q install transformers datasets scikit-learn seaborn tqdm tensorboard pyyaml pandas pillow matplotlib opencv-python



In [ ]:
# ============================================================
# STEP 3: Clone Project from GitHub
# ============================================================
import os, shutil, subprocess

# Your repository
GITHUB_REPO = 'https://github.com/au510621104021/FFake_detection.git'
GITHUB_BRANCH = ''  # keep empty to use repo default branch; or set e.g. 'main'

# Optional for private repos: set token in Colab Secrets or env var GITHUB_TOKEN
# from google.colab import userdata
# os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '').strip()

clone_url = GITHUB_REPO
if GITHUB_TOKEN and GITHUB_REPO.startswith('https://'):
    clone_url = GITHUB_REPO.replace('https://', f'https://{GITHUB_TOKEN}@', 1)

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

clone_cmd = ['git', 'clone', '--single-branch']
if GITHUB_BRANCH.strip():
    clone_cmd += ['--branch', GITHUB_BRANCH.strip()]
clone_cmd += [clone_url, PROJECT_DIR]

subprocess.check_call(clone_cmd)

# Remove token from local git config after clone for safety
if clone_url != GITHUB_REPO:
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'remote', 'set-url', 'origin', GITHUB_REPO])

commit = subprocess.check_output(['git', '-C', PROJECT_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip()
branch = subprocess.check_output(['git', '-C', PROJECT_DIR, 'rev-parse', '--abbrev-ref', 'HEAD'], text=True).strip()
print(f'Cloned: {GITHUB_REPO}')
print(f'Branch: {branch}')
print(f'Commit: {commit}')



In [ ]:
# ============================================================
# STEP 3.1: Install Project Requirements (from cloned repo)
# ============================================================
import os, subprocess, sys

req_file = os.path.join(PROJECT_DIR, 'requirements.txt')
if not os.path.exists(req_file):
    raise FileNotFoundError(f'requirements.txt not found at {req_file}')

# Use your repo dependencies directly
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req_file])
print('Installed requirements from repo')

# Optional: update later without recloning
# !git -C /content/fake_news_project pull --ff-only



In [ ]:
# ============================================================
# STEP 4: Dataset Setup (ISOT or custom CSV)
# ============================================================
import os
from google.colab import files as gfiles

DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'isot')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Data directory: {DATA_DIR}')
existing = sorted(os.listdir(DATA_DIR))
print('Existing files:', existing if existing else 'None')

need_isot = not (os.path.exists(os.path.join(DATA_DIR, 'Fake.csv')) and os.path.exists(os.path.join(DATA_DIR, 'True.csv')))

if need_isot:
    print('
Upload dataset files now.')
    print('For ISOT: Fake.csv + True.csv')
    print('For custom data: train.csv/test.csv or dataset.csv with text + label columns')
    up = gfiles.upload()

    for name, data in up.items():
        dest = os.path.join(DATA_DIR, name)
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  Saved: {dest} ({len(data)/1e6:.1f} MB)')

print('
Final files in data dir:', sorted(os.listdir(DATA_DIR)))



In [ ]:
# ============================================================
# STEP 5: Verify Project Structure
# ============================================================
import sys, os
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# Quick structure check
required_paths = ['src', 'config/config.yaml', 'scripts/train.py']
missing = [p for p in required_paths if not os.path.exists(os.path.join(PROJECT_DIR, p))]
if missing:
    raise FileNotFoundError(f'Missing required project files: {missing}')

print('Project looks valid. Root:', PROJECT_DIR)
for root, dirs, fls in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in {'__pycache__', '.git', '.venv', '.venv310', 'node_modules'}]
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    sub = '  ' * (level + 1)
    for f in fls[:8]:
        print(f'{sub}{f}')
    if level >= 2:
        continue



In [ ]:
# ============================================================
# STEP 6: Load & Adjust Config for Colab
# ============================================================
import yaml, os
config_path = os.path.join(PROJECT_DIR, 'config', 'config.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

# Decide adapter from uploaded files
has_isot = os.path.exists(os.path.join(DATA_DIR, 'Fake.csv')) and os.path.exists(os.path.join(DATA_DIR, 'True.csv'))
has_generic = any(os.path.exists(os.path.join(DATA_DIR, n)) for n in ['dataset.csv', 'data.csv', 'train.csv', 'test.csv'])

if has_isot:
    config['data']['dataset_name'] = 'isot'
elif has_generic:
    config['data']['dataset_name'] = 'generic'
else:
    raise ValueError(
        f'No usable dataset files in {DATA_DIR}. '        'Upload Fake.csv+True.csv (ISOT) or dataset/train/test CSV files.'
    )

# Runtime-aware Colab overrides
use_cuda = bool(globals().get('USE_CUDA', False))
if use_cuda:
    config['training']['batch_size'] = 8
    config['training']['num_epochs'] = 10
    config['training']['fp16'] = True
    config['data']['num_workers'] = 2
    config['data']['pin_memory'] = True
else:
    config['training']['batch_size'] = 2
    config['training']['num_epochs'] = 3
    config['training']['fp16'] = False
    config['data']['num_workers'] = 0
    config['data']['pin_memory'] = False
    # Speed up CPU runs to avoid long runtimes
    config['model']['text_encoder']['max_length'] = min(128, config['model']['text_encoder'].get('max_length', 128))

config['data']['data_dir'] = DATA_DIR

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('=== Colab Training Config ===')
print(f"  Runtime    : {'GPU' if use_cuda else 'CPU'}")
print(f"  Dataset    : {config['data']['dataset_name']}")
print(f"  Data dir   : {config['data']['data_dir']}")
print(f"  Batch size : {config['training']['batch_size']}")
print(f"  Epochs     : {config['training']['num_epochs']}")
print(f"  Max length : {config['model']['text_encoder']['max_length']}")
print(f"  FP16       : {config['training']['fp16']}")



In [ ]:
# ============================================================
# STEP 7: Train the Model
# ============================================================
import random, numpy as np, torch
from src.models import MultimodalFakeNewsDetector
from src.data.dataset import get_dataloader
from src.training.trainer import Trainer

# Seed
seed = config['training'].get('seed', 42)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# DataLoaders
dataloaders = get_dataloader(
    data_dir=config['data']['data_dir'],
    dataset_name=config['data']['dataset_name'],
    tokenizer_name=config['model']['text_encoder']['name'],
    max_length=config['model']['text_encoder']['max_length'],
    image_size=config['model']['image_encoder']['image_size'],
    batch_size=config['training']['batch_size'],
    val_split=config['training'].get('val_split', 0.15),
    test_split=config['training'].get('test_split', 0.15),
    num_workers=config['data'].get('num_workers', 0),
    pin_memory=config['data'].get('pin_memory', False),
    seed=seed,
)

# Model
model = MultimodalFakeNewsDetector(config)
params = model.get_trainable_params()
print(f"Params: {params['trainable']:,} trainable / {params['total']:,} total ({params['trainable_pct']:.1f}%)")

# Train
trainer = Trainer(model=model, config=config, device=device)
MODE = 'multimodal'  # Change to 'text_only' or 'image_only' for ablation
result = trainer.train(
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    mode=MODE,
)
print(f"\nBest epoch: {result['best_epoch']}  |  Best F1: {result['best_val_f1']:.4f}")



In [ ]:
# ============================================================
# STEP 8: Evaluate on Test Set
# ============================================================
print('Running final evaluation on test set...')
test_metrics = trainer.evaluate(
    test_loader=dataloaders['test'],
    mode=MODE,
    generate_plots=True,
)

# Show plots inline
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from pathlib import Path
plots_dir = Path(config.get('logging',{}).get('log_dir','./logs')) / 'metrics'
for img in sorted(plots_dir.glob('*.png')):
    print(f'\n--- {img.name} ---')
    display(IPImage(filename=str(img)))

In [ ]:
# ============================================================
# STEP 9: Plot Training History
# ============================================================
import matplotlib.pyplot as plt

history = result['history']
epochs = [h['epoch'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs, [h['train_loss'] for h in history], 'b-o', label='Train')
axes[0].plot(epochs, [h['val_loss'] for h in history], 'r-o', label='Val')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [h['train_accuracy'] for h in history], 'b-o', label='Train')
axes[1].plot(epochs, [h['val_accuracy'] for h in history], 'r-o', label='Val')
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

# F1
axes[2].plot(epochs, [h['train_f1'] for h in history], 'b-o', label='Train')
axes[2].plot(epochs, [h['val_f1'] for h in history], 'r-o', label='Val')
axes[2].set_title('F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 10: Download Trained Model
# ============================================================
from google.colab import files
import shutil

# Zip checkpoints + results
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints')
results_dir = os.path.join(PROJECT_DIR, 'results')
archive = '/content/trained_model'
dirs_to_zip = [d for d in [ckpt_dir, results_dir] if os.path.exists(d)]

os.makedirs('/content/export', exist_ok=True)
for d in dirs_to_zip:
    dest = os.path.join('/content/export', os.path.basename(d))
    if os.path.exists(dest): shutil.rmtree(dest)
    shutil.copytree(d, dest)

shutil.make_archive(archive, 'zip', '/content/export')
print('Downloading trained_model.zip...')
files.download(archive + '.zip')

In [ ]:
# ============================================================
# STEP 11 (Optional): Ablation Study
# ============================================================
# Uncomment to compare multimodal vs text-only vs image-only

# from src.training.metrics import compare_models
# results_all = {}
# for mode in ['multimodal', 'text_only', 'image_only']:
#     print(f'\n=== {mode.upper()} ===')
#     m = trainer.evaluate(dataloaders['test'], mode=mode, generate_plots=False)
#     results_all[mode] = m
#     print(f"  Acc: {m['accuracy']:.4f}  F1: {m['f1']:.4f}")
# compare_models(results_all, save_dir='./results')